# 选修E10 · Day 3 上机：Agent生态与治理--平台设计与市场监管

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 解释Agent平台三边市场模型和四类网络效应（含AI特有的"数据飞轮"）
2. 用 **pydantic** 定义四种治理规则schema契约（准入/分润/惩罚/信誉），实现结构化输出
3. 用 **networkx** 构建Agent生态网络，做核心-边缘/中心性分析
4. 用 **mesa** 多Agent仿真（30 agents/15 ticks）对比不同治理规则下的生态健康（Gini/成交/欺诈率）
5. 用 **numpy-financial** 做平台12月NPV估值，量化治理规则对平台价值的影响
6. 建立天道推演×生态治理沙盘同构认知--用三时间线推演不同治理规则在MCP/A2A生态演化下的走向

## 真实库与真实数据
- **networkx**（生态网络拓扑）：https://networkx.org/documentation/stable/
- **mesa**（多Agent仿真）：https://github.com/projectmesa/mesa
- **pydantic**（治理schema契约）：https://github.com/pydantic/pydantic
- **numpy-financial**（平台估值NPV/IRR）：https://github.com/numpy/numpy-financial
- **真实Agent生态案例**：A2A协议/MCP生态/Coze/Dify/LangGraph/GPT Store/Hugging Face Spaces

> 所有库与数据均来自官方公开源，不需要API Key。mesa仿真保持小规模（30 agents/15 ticks/<10s）。

## 0. 环境准备

> 所有库（networkx/mesa/pydantic/numpy-financial/pandas/matplotlib/numpy）均为本地可用库，不需要API Key。

In [ ]:
# !pip install networkx mesa pydantic numpy-financial pandas matplotlib numpy -q

import warnings
warnings.filterwarnings('ignore')

import networkx as nx
from mesa import Model, Agent
from mesa.datacollection import DataCollector
from pydantic import BaseModel, Field, model_validator
import numpy_financial as npf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal, Optional

print(f"✓ networkx {nx.__version__}, mesa, pydantic, numpy-financial, pandas {pd.__version__}, numpy {np.__version__} 已就绪")
print("  networkx: Agent生态网络拓扑分析")
print("  mesa: 多Agent仿真（小规模，30 agents/15 ticks）")
print("  pydantic: 治理规则schema契约")
print("  numpy-financial: 平台12月NPV估值")

## 1. 真实Agent生态治理参数

本Day的参数基于真实Agent生态平台治理实践：

| 参数 | 严准入+高分润 | 宽准入+低分润 | 真实来源 |
|------|------------|------------|---------|
| 准入通过率 | 0.40 | 0.85 | App Store严审（~40%）vs HF开放（~85%） |
| 平台抽成率 | 25% | 5% | GPT Store 30%/15% vs MCP 0%（取中间值） |
| 欺诈惩罚力度 | 0.80 | 0.30 | 严治平台惩罚重 |
| 初始Agent数 | 30 | 30 | 教学小规模 |
| 仿真ticks | 15 | 15 | <10s跑完 |
| 初始投资 | $8000 | $8000 | 平台开发部署 |
| 月贴现率 | 0.10/12 | 0.10/12 | 年化10% |

**真实Agent生态案例**（来自各官方文档）：
- A2A协议（Google）: https://github.com/google/A2A
- MCP生态（Anthropic）: https://modelcontextprotocol.io/
- Coze（字节）: https://www.coze.com/
- Dify: https://dify.ai/
- LangGraph: https://langchain.ai/
- OpenAI GPT Store: https://openai.com/chatgpt/pricing/
- Hugging Face Spaces: https://huggingface.co/

In [ ]:
# 真实Agent生态治理参数（可追溯来源）
# 来源1: A2A协议 https://github.com/google/A2A （开放协议）
# 来源2: MCP生态 https://modelcontextprotocol.io/ （0抽成）
# 来源3: GPT Store抽成 https://openai.com/chatgpt/pricing/ （30%/15%）
# 来源4: Hugging Face https://huggingface.co/ （0抽成开源）
# 来源5: Coze https://www.coze.com/ , Dify https://dify.ai/

# === 两种治理规则对比参数 ===
GOVERNANCE_RULES = {
    "strict_high_share": {
        "name": "严准入+高分润",
        "admission_rate": 0.40,       # 严准入，App Store风格
        "platform_share": 0.25,       # 25%抽成，GPT Store中间值
        "fraud_penalty": 0.80,        # 严惩欺诈
        "initial_reputation": 50.0,
        "description": "严准入（40%通过）+高分润（25%抽成）+严惩欺诈"
    },
    "open_low_share": {
        "name": "宽准入+低分润",
        "admission_rate": 0.85,       # 宽准入，HF风格
        "platform_share": 0.05,       # 5%抽成，接近MCP 0%
        "fraud_penalty": 0.30,        # 轻惩欺诈
        "initial_reputation": 50.0,
        "description": "宽准入（85%通过）+低分润（5%抽成）+轻惩欺诈"
    }
}

# === 仿真参数（小规模，<10s） ===
N_AGENTS = 30        # 30个Agent（开发者Agent + 用户Agent）
N_TICKS = 15         # 15 ticks
SEED = 42            # 固定随机种子，可复现

# === 平台估值参数 ===
INITIAL_INVESTMENT = 8000.0       # 平台开发部署初始投资
MONTHLY_DISCOUNT_RATE = 0.10 / 12 # 年化10%月贴现率
FORECAST_MONTHS = 12              # 12月预测

print("=== 两种治理规则对比参数 ===")
for k, v in GOVERNANCE_RULES.items():
    print(f"{v['name']}: 准入率={v['admission_rate']}, 抽成={v['platform_share']*100:.0f}%, 惩罚={v['fraud_penalty']}")
print(f"\n仿真规模: {N_AGENTS} agents × {N_TICKS} ticks (seed={SEED})")
print(f"平台估值: 初始投资${INITIAL_INVESTMENT}, 月贴现率{MONTHLY_DISCOUNT_RATE:.4f}, {FORECAST_MONTHS}月预测")

## 2. TODO 1：pydantic四种治理规则schema定义

**四种Agent平台治理规则契约**：

| 规则 | 字段 | 治理逻辑 |
|------|------|---------|
| 准入 | admission_rate, review_level | 控制谁能加入生态 |
| 分润 | platform_share, developer_share | 平台与开发者如何分钱 |
| 惩罚 | fraud_penalty, violation_threshold | 违规怎么罚 |
| 信誉 | initial_score, decay_rate, weight | 信誉怎么算和衰减 |

**要求**：
- 用pydantic BaseModel定义四种治理规则
- 每种规则实现 `validate_rule()` 方法验证约束
- 每种规则实现 `to_contract()` 方法导出结构化输出（Agent可发现的治理声明）
- 用 `@model_validator` 验证字段约束（概率0-1、分润和=1）

**理论连接**：pydantic schema不仅是数据验证，更是API Economy 2.0的"Agent可发现治理声明"--Agent通过读取平台的治理schema，自动判断"我能加入哪个平台、被怎么治理、违规怎么罚"。

In [ ]:
# TODO 1：pydantic四种治理规则schema定义
# 提示：继承BaseModel
#   AdmissionRule: admission_rate (0,1), review_level in {"self","platform","third_party"}
#   RevenueShare: platform_share [0,1], developer_share [0,1], 验证 platform_share+developer_share=1
#   PenaltyRule: fraud_penalty (0,1], violation_threshold (0,1)
#   ReputationScoring: initial_score (0,100], decay_rate [0,1], weight (0,1]
#   每种实现 to_contract() 方法（导出Agent可发现的治理声明）
#   用 @model_validator 验证跨字段约束

# ===== 你的代码 =====

# raise NotImplementedError

## 3. TODO 2：networkx构建Agent生态网络

**真实Agent生态网络结构**（基于真实平台和公司构建）：

| 节点类型 | 节点数 | 示例 |
|---------|-------|------|
| Platform | 7 | MCP, A2A, Coze, Dify, LangGraph, GPT Store, HF Spaces |
| Developer | 7 | OpenAI, Anthropic, Google, Meta, Mistral, ByteDance, LangChain |
| ToolProvider | 4 | GitHub, Slack, Notion, Stripe |
| User | 3 | Enterprise, Individual, Research |

**边类型**：PUBLISHES_ON（开发者→平台）、USES_AGENT（用户→平台）、MCP_INTEGRATES（工具→平台）、A2A_CALLS（开发者→开发者，A2A协议跨平台调用）

**要求**：
- 用 `nx.MultiDiGraph()` 构建Agent生态网络
- 添加4类节点（带node_type属性）
- 添加4类边（带relation属性）
- 打印节点/边统计

**理论连接**：networkx构建的生态网络反映真实Agent生态结构。MCP_INTEGRATES和A2A_CALLS边是2026新型关系，体现MCP协议和A2A协议带来的生态互联。

In [ ]:
# TODO 2：networkx构建Agent生态网络
# 提示：
#   1. G = nx.MultiDiGraph()
#   2. 添加7个Platform节点 (MCP, A2A, Coze, Dify, LangGraph, GPT Store, HF Spaces)
#   3. 添加7个Developer节点 (OpenAI, Anthropic, Google, Meta, Mistral, ByteDance, LangChain)
#   4. 添加4个ToolProvider节点 (GitHub, Slack, Notion, Stripe)
#   5. 添加3个User节点 (Enterprise, Individual, Research)
#   6. 添加边：PUBLISHES_ON / USES_AGENT / MCP_INTEGRATES / A2A_CALLS
#   7. 打印节点数、边数、按relation分组的边数
# 要求：构建真实生态网络，打印统计

# ===== 你的代码 =====

# raise NotImplementedError

## 4. TODO 3：networkx生态拓扑分析

用networkx图算法分析Agent生态网络的拓扑特征。

**核心指标**：
- **度分布**：入度/出度的均值/最大/最小，反映生态参与度
- **聚类系数**：节点间互联程度，反映生态紧密性
- **核心-边缘结构**：`nx.core_number` 划分核心/边缘节点
- **中心性**：degree centrality / betweenness centrality / closeness centrality

**要求**：
- 计算度分布
- 计算聚类系数（需转为无向图）
- 计算核心-边缘结构
- 计算三种中心性，找出生态"枢纽"节点
- 打印分析结果

**理论连接**：核心-边缘结构识别"谁在生态核心、谁是单点故障风险"。中心性指标反映"谁是生态枢纽"。这是天道推演的"局势感知"能力--识别生态的关键节点。

In [ ]:
# TODO 3：networkx生态拓扑分析
# 提示：
#   1. in_degrees = dict(G.in_degree()), out_degrees = dict(G.out_degree())
#   2. G_undirected = nx.Graph() 合并多重边为权重
#   3. clustering = nx.clustering(G_undirected, weight='weight')
#   4. core_numbers = nx.core_number(G_undirected)
#   5. 三种中心性：nx.degree_centrality, nx.betweenness_centrality, nx.closeness_centrality
#   6. 划分核心/边缘节点 (core_number >= max_core 为核心)
# 要求：计算所有指标，打印度分布/聚类/核心-边缘/中心性Top5

# ===== 你的代码 =====

# raise NotImplementedError

## 5. TODO 4：mesa多Agent仿真（小规模，30 agents/15 ticks）

用 **mesa** 多Agent仿真模拟平台治理规则对生态健康的影响。

**仿真设计**（小规模，<10s跑完）：
- **PlatformAgent**（1个）：执行治理规则（准入/分润/惩罚/信誉更新）
- **DevAgent**（20个）：开发Agent，积累信誉，可能欺诈
- **UserAgent**（10个）：调用Agent，按信誉选择
- **DataCollector**：每tick收集 Gini系数/成交率/欺诈率/平台收入

**Gini系数公式**（0-indexed）：
`G = sum((2*i - n + 1) * x_i) / (n * sum(x_i))`，其中x已排序

**两种治理规则对比**：
- 严准入+高分润：admission_rate=0.4, platform_share=0.25, fraud_penalty=0.80
- 宽准入+低分润：admission_rate=0.85, platform_share=0.05, fraud_penalty=0.30

**要求**：
- 实现PlatformAgent/DevAgent/UserAgent三类Agent
- 实现AgentEcosystemModel，初始化时按治理规则配置
- 跑15 ticks，每tick收集指标
- 对比两种治理规则下的Gini/成交/欺诈率/平台收入

**理论连接**：mesa多Agent仿真是天道推演的代码化版本--在沙盘中让Agent按规则互动，观察宏观涌现。这是天道推演的"沙盘模拟"能力。

In [ ]:
# TODO 4：mesa多Agent仿真（小规模，30 agents/15 ticks）
# 提示：
#   1. def compute_gini(x): 0-indexed公式 G = sum((2*i-n+1)*x_i) / (n*sum(x_i))
#   2. class DevAgent(Agent): step()中按治理规则决定是否欺诈，更新信誉
#   3. class UserAgent(Agent): step()中按信誉选择DevAgent调用，产生成交
#   4. class PlatformAgent(Agent): step()中执行惩罚/分润/准入
#   5. class AgentEcosystemModel(Model): __init__按治理规则配置，DataCollector收集4个指标
#   6. model.agents.shuffle_do("step") + collector.collect(self)
#   7. 对比两种治理规则，跑15 ticks
# 要求：实现仿真，打印两种治理规则下的最终Gini/成交/欺诈率/平台收入对比

# ===== 你的代码 =====

# raise NotImplementedError

## 6. TODO 5：numpy-financial平台估值 + 治理规则效果量化

用 **numpy-financial** 对平台做12月NPV估值，量化治理规则选择对平台长期价值的影响。

**建模假设**（基于TODO4仿真结果）：
- 严准入+高分润：月活跃Agent 25，月成交额/Agent $80，平台抽成25%，月运营成本$500，月增长5%
- 宽准入+低分润：月活跃Agent 18，月成交额/Agent $45，平台抽成5%，月运营成本$300，月增长8%

**12月现金流**：t=0为初始投资（-$8000），t=1..12为月净收入（成交 × 抽成 - 运营成本，含增长）

**要求**：
- 建模12月现金流
- 用 `npf.npv(rate, cashflows)` 计算NPV
- 用 `npf.irr(cashflows)` 计算IRR
- 对比两种治理规则的平台估值
- 分析"高分润 vs 高规模"的治理权衡

**理论连接**：numpy-financial平台估值是"治理规则→生态健康→平台价值"因果链的最后一步。NPV对比揭示治理规则选择的长期财务影响。这是天道推演的"最优路径推荐"能力。

In [ ]:
# TODO 5：numpy-financial平台估值 + 治理规则效果量化
# 提示：
#   1. 两种治理规则的月活跃Agent/月成交/抽成/运营成本/增长率不同
#   2. 月净收入 = 月活跃Agent * 月成交/Agent * 抽成 - 运营成本
#   3. 12月现金流，每月按增长率增长
#   4. npf.npv(MONTHLY_DISCOUNT_RATE, cashflows) 计算NPV
#   5. npf.irr(cashflows) 计算IRR
#   6. 对比两种治理规则的NPV/IRR/总利润
# 要求：计算两种治理规则的12月NPV/IRR，打印对比表

# ===== 你的代码 =====

# raise NotImplementedError

## 7. TODO 6：matplotlib可视化（4个子图）

用matplotlib绘制4个子图：

1. **Agent生态网络拓扑**（networkx布局）：节点按类型着色，边按relation着色，标注核心节点
2. **治理规则效果对比**（柱状图）：两种治理规则的Gini/成交/欺诈率/平台收入4个指标对比
3. **仿真Gini演化曲线**（折线图）：两种治理规则15 ticks的Gini系数演化
4. **中心性分布**（横向柱状图）：Top 8节点的betweenness centrality

**理论连接**：可视化让生态治理的对比直观可见，是天道推演沙盘的"局势可视化"。

In [ ]:
# TODO 6：matplotlib可视化（4个子图）
# 提示：
#   1. plt.subplots(2,2,figsize=(14,10))
#   2. 子图1: nx.spring_layout + nx.draw_networkx_nodes/edges/labels，按node_type着色
#   3. 子图2: ax.bar 对比两种治理规则的4个指标
#   4. 子图3: ax.plot 两种治理规则的Gini演化曲线
#   5. 子图4: ax.barh Top 8节点的betweenness centrality
# 要求：4个子图都有标题、标签、数据

# ===== 你的代码 =====

# raise NotImplementedError

## 8. 天道推演 × 生态治理沙盘

本Day的生态治理设计本质是**商业版的天道推演沙盘**：

| 天道推演能力 | 生态治理设计对应 | 产出 |
|-------------|----------------|------|
| 局势感知 | 真实Agent生态案例 + networkx拓扑分析 | 生态基线 |
| 因果链追踪 | 治理规则 → 生态健康 → 平台价值 | 因果模型 |
| 沙盘模拟（3层推演） | mesa 15 ticks / numpy-financial 12月NPV / MCP+A2A 3年演化 | 三时间线推演 |
| 概率评估 | 仿真Gini分布 + NPV对比 | 风险量化 |
| 最优路径推荐 | 两种治理规则对比 + 平台估值 | 策略选择 |

### 三时间线推演

- **immediate（tick，秒级）**：mesa仿真15 ticks，治理规则→Agent行为→生态指标
- **near（年，12月）**：numpy-financial NPV/IRR，治理规则→现金流→平台估值
- **far（3年+）**：MCP协议标准化 + A2A经济兴起 + 数据飞轮成熟

### 2026-2028生态演化预判

| 时间 | 主流治理 | 主流平台 | 触发条件 |
|------|---------|---------|---------|
| 2026 | 平台集中治理（GPT Store/Coze） | GPT Store/Coze/Dify | 平台抽成25-30% |
| 2027 | 协议化治理（MCP/A2A）兴起 | MCP生态/A2A协议 | 开放协议降低抽成至0-5% |
| 2028 | 联邦治理（跨平台信誉） | 多平台共存 | A2A协议成熟、信誉跨平台 |

---

## 9. 作业与评估

- [ ] 完成 `starter.ipynb`（6个TODO全部填好）
- [ ] pydantic治理schema验证通过
- [ ] networkx生态分析有数据（度分布/聚类/核心-边缘/中心性）
- [ ] mesa仿真两种治理规则对比有数据（Gini/成交/欺诈率/平台收入）
- [ ] numpy-financial平台NPV估值有结果
- [ ] 4个子图有数据
- [ ] 一段300字分析：严准入+高分润 vs 宽准入+低分润，哪种治理规则更优？为什么？

---

*本笔记本由v5.0学习材料包升级生成。理论部分引用独立教材，上机部分用真实库（networkx+mesa+pydantic+numpy-financial+pandas+matplotlib+numpy）+ TODO脚手架，Agent生态案例基于真实公开数据。*